In [1]:
import random
import math
import pandas as pd
import numpy as np
import time

from preferences import prefs
from user import user_prefs, experiments

startPoint = [59.927085, 30.317504]
endPoint = [59.935408, 30.327108]

POPULATION_SIZE = 200
GENERATIONS = 100

MIN_ROUTE_POINTS = 2
MAX_ROUTE_POINTS = 15

MUTATION_RATE = 0.2
TOURNAMENT_SIZE = 5

PRINT = False

# Особь:
# {
#     "route": [индексы точек из df],
#     "fitness": число
# }


In [2]:
df = pd.read_csv('data/places.csv')

**ИЗМЕНЕНИЕ ОСОБИ**

In [3]:
# Кроссовер
def crossover(parent1, parent2):

    route1 = parent1["route"]
    route2 = parent2["route"]

    if len(route1) < 2 or len(route2) < 2:
        return {
            "route": route1.copy(),
            "fitness": None
        }

    cut1 = random.randint(1, len(route1) - 1)
    cut2 = random.randint(1, len(route2) - 1)

    child_route = route1[:cut1]

    for point in route2[cut2:]:
        if point not in child_route:
            child_route.append(point)

    return {
        "route": child_route,
        "fitness": None
    }


# Мутация
def mutate(individual, df):

    route = individual["route"][:]

    if random.random() > MUTATION_RATE:
        return individual

    mutation_type = random.choice([
        "swap",
        "remove",
        "add"
    ])

    if mutation_type == "swap" and len(route) >= 2:

        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]

    elif mutation_type == "remove":

        if len(route) > MIN_ROUTE_POINTS:
            idx = random.randint(0, len(route) - 1)
            route.pop(idx)

    elif mutation_type == "add":

        if len(route) < MAX_ROUTE_POINTS:

            available = list(
                set(range(len(df))) - set(route)
            )

            if available:

                count_to_add = random.randint(1, 3)

                for _ in range(count_to_add):

                    if not available:
                        break

                    new_point = random.choice(available)
                    available.remove(new_point)

                    insert_pos = random.randint(
                        0,
                        len(route)
                    )

                    route.insert(insert_pos, new_point)

    individual["route"] = route
    individual["fitness"] = None

    return individual

**ЭВОЛЮЦИОННЫЕ АЛГОРИТМЫ**

In [4]:
# Интерес конкретной точки
def point_interest_score(point_row, unique_user_prefs):
    score = 0
    for category, tags in prefs.items():
        weight = unique_user_prefs[category]
        for tag in tags:
            if tag in point_row.index and point_row[tag]:
                # print("point_interest_score")
                # print(category, weight, tag, point_row)
                score += weight

    return score


# Прямое расстояние между точками в метрах
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))


# Длина маршрута
def route_distance(route, df):

    total_distance = 0

    prev_lat, prev_lon = startPoint

    for idx in route:
        point = df.iloc[idx]

        total_distance += haversine(
            prev_lat,
            prev_lon,
            point["lat"],
            point["lon"]
        )

        prev_lat = point["lat"]
        prev_lon = point["lon"]

    total_distance += haversine(
        prev_lat,
        prev_lon,
        endPoint[0],
        endPoint[1]
    )

    return total_distance


# Фитнес функция
def fitness_function(route, df, unique_user_prefs):

    total_interest = 0

    for idx in route:

        point = df.iloc[idx]

        total_interest += point_interest_score(point, unique_user_prefs)

    distance = route_distance(route, df)

    fitness = (
        total_interest * 20
        - distance * 0.03
    )

    return fitness


# Генерация случайной особи
def create_individual(df):

    route_size = random.randint(
        MIN_ROUTE_POINTS,
        MAX_ROUTE_POINTS
    )

    route = random.sample(
        range(len(df)),
        route_size
    )

    return {
        "route": route,
        "fitness": None
    }


# Создание популяции
def create_population(df):
    return [
        create_individual(df)
        for _ in range(POPULATION_SIZE)
    ]


# Оценка популяции
def evaluate_population(population, df, unique_user_prefs):

    for individual in population:
        individual["fitness"] = fitness_function(
            individual["route"],
            df,
            unique_user_prefs
        )


# Турнир
def tournament_selection(population):

    tournament = random.sample(
        population,
        TOURNAMENT_SIZE
    )

    tournament.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    return tournament[0]


# Создание нового поколения
def create_next_generation(population, df):

    new_population = []

    population.sort(
        key=lambda x: x["fitness"],
        reverse=True
    )

    elite = population[:5]

    new_population.extend(elite)

    while len(new_population) < POPULATION_SIZE:

        parent1 = tournament_selection(population)
        parent2 = tournament_selection(population)

        child = crossover(parent1, parent2)

        child = mutate(child, df)

        new_population.append(child)

    return new_population


# Генетический алгоритм
def genetic_algorithm(df, unique_user_prefs):
    print(unique_user_prefs)

    population = create_population(df)

    for generation in range(GENERATIONS):

        evaluate_population(population, df, unique_user_prefs)

        best = max(
            population,
            key=lambda x: x["fitness"]
        )

        if PRINT:
            print(
                f"Поколение {generation} | "
                f"Лучшая фитнес-функция: {best['fitness']:.10f} | "
                f"Длина маршрута: {len(best['route'])}"
            )

        population = create_next_generation(
            population,
            df
        )

    evaluate_population(population, df, unique_user_prefs)

    best = max(
        population,
        key=lambda x: x["fitness"]
    )

    return best

**ОБРАБОТКА**

In [5]:
# Вычисление проекции каждой точки на всю длину маршрута (прямую)
def project_point_to_line(point, start, end):
    """
    Проекция точки на линию START -> END.
    Возвращает параметр t:
    0   = начало линии
    1   = конец линии
    0.5 = середина линии
    >1  = дальше конца
    <0  = до начала
    """

    px, py = point
    sx, sy = start
    ex, ey = end

    line_vec = np.array([ex - sx, ey - sy])
    point_vec = np.array([px - sx, py - sy])

    line_len_sq = np.dot(line_vec, line_vec)

    if line_len_sq == 0:
        return 0

    t = np.dot(point_vec, line_vec) / line_len_sq

    return t

# Полный маршрут, сортировка точек в зависимости от их положения между собой
def build_full_route(best_individual, df):

    route_points = []

    points = []

    for idx in best_individual["route"]:

        point = df.iloc[idx]

        lat = float(point["lat"])
        lon = float(point["lon"])

        t = project_point_to_line(
            [lat, lon],
            startPoint,
            endPoint
        )

        points.append({
            "name": point["name"],
            "lat": lat,
            "lon": lon,
            "t": t
        })

    points.sort(key=lambda x: x["t"])

    route_points.append({
        "name": "START",
        "lat": startPoint[0],
        "lon": startPoint[1]
    })

    route_points.extend(points)

    route_points.append({
        "name": "END",
        "lat": endPoint[0],
        "lon": endPoint[1]
    })

    return route_points

# Ссылка на яндекс карты
def generate_yandex_maps_url(route_points):

    rtext = "~".join(
        f"{point['lat']},{point['lon']}"
        for point in route_points
    )

    yandex_url = (
        f"https://yandex.ru/maps/"
        f"?mode=routes"
        f"&rtext={rtext}"
        f"&rtt=pd"
    )

    return yandex_url


**ЗАПУСК ОДНОЙ ГЕНЕРАЦИИ**

In [6]:
# Запуск

# best_individual = genetic_algorithm(df)

# final_route = build_full_route(
#     best_individual,
#     df
# )

# print("СПИСОК МЕСТ")

# places_names = []

# for point in final_route:

#     # START / END можно исключить
#     if point["name"] not in ["START", "END"]:
#         places_names.append(point["name"])

# for i, name in enumerate(places_names):

#     print(f"{i + 1}. {name}")


# yandex_url = generate_yandex_maps_url(
#     final_route
# )

# print("ССЫЛКА НА МАРШРУТ")

# print(yandex_url)

**ЭКСПЕРИМЕНТЫ**

In [7]:
# Интересность маршрута
def route_interest_score(route, df, unique_user_prefs):
    total = 0

    for idx in route:
        point = df.iloc[idx]
        total += point_interest_score(point, unique_user_prefs)

    return total

# Запуск одного эксперимента
def run_experiment(df, unique_user_prefs, experiment_name="exp"):

    start_time = time.time()

    best_individual = genetic_algorithm(df, unique_user_prefs)

    final_route = build_full_route(best_individual, df)

    end_time = time.time()
    
    interest = route_interest_score(best_individual["route"], df, unique_user_prefs)
    distance = route_distance(best_individual["route"], df)
    duration = end_time - start_time

    yandex_url = generate_yandex_maps_url(final_route)

    route_text = route_to_names(final_route)

    return {
        "experiment": experiment_name,
        "interest_score": interest,
        "distance": distance,
        "time_sec": duration,
        "route_length": len(best_individual["route"]),
        "route_text": route_text,
        "yandex_url": yandex_url
    }


# Запуск множества экспериментов
def run_multiple_experiments(df, experiments):

    results = []

    for exp in experiments:

        print(f"\nRunning: {exp['name']}")

        result = run_experiment(
            df=df,
            unique_user_prefs=exp["user_prefs"],
            experiment_name=exp["name"]
        )

        results.append(result)

        print(f"Done: {exp['name']} | time: {result['time_sec']:.2f}s")

    return results


# Вывод точек маршрута текстом
def route_to_names(final_route):

    names = []

    for point in final_route:
        if point["name"] not in ["START", "END"]:
            names.append(point["name"])


    return " -> ".join(names)


# Сохранение результата
def save_results(results, filename="ga_experiments.csv"):

    rows = []

    for r in results:

        rows.append({
            "experiment": r["experiment"],
            "interest_score": r["interest_score"],
            "distance": r["distance"],
            "time_sec": r["time_sec"],
            "route_length": r["route_length"],
            "route_text": r["route_text"],
            "yandex_url": r["yandex_url"]
        })

    df_results = pd.DataFrame(rows)
    df_results.to_csv(filename, index=False)

    print(f"\nSaved to {filename}")

results = run_multiple_experiments(df, experiments)

save_results(results)


Running: user_1_1_military_focus
{'military': 5, 'religion': 1, 'architecture': 1, 'transport': 3, 'sight': 1, 'interactive': 2, 'nutrition': 0, 'housing': 0}
Done: user_1_1_military_focus | time: 46.19s

Running: user_1_2_military_focus
{'military': 5, 'religion': 1, 'architecture': 1, 'transport': 3, 'sight': 1, 'interactive': 2, 'nutrition': 0, 'housing': 0}
Done: user_1_2_military_focus | time: 45.04s

Running: user_1_3_military_focus
{'military': 5, 'religion': 1, 'architecture': 1, 'transport': 3, 'sight': 1, 'interactive': 2, 'nutrition': 0, 'housing': 0}
Done: user_1_3_military_focus | time: 45.24s

Running: user_1_4_military_focus
{'military': 5, 'religion': 1, 'architecture': 1, 'transport': 3, 'sight': 1, 'interactive': 2, 'nutrition': 0, 'housing': 0}
Done: user_1_4_military_focus | time: 42.12s

Running: user_2_1_religion_focus
{'military': 1, 'religion': 5, 'architecture': 5, 'transport': 1, 'sight': 5, 'interactive': 1, 'nutrition': 0, 'housing': 0}
Done: user_2_1_relig